# 06 — Checks: attack-zone validation, robustness, multiple comparisons

Added 2026-10-01, after the main analyses. Nothing here changes a pre-registered result. It shows how far those
results can be trusted.
1. **Attack-zone validation:** do our computed heart/shadow/chase/waste bands reproduce Savant's published run values by zone?
2. **Robustness:** do the H1 and velocity results hold under the alternatives rejected during design?
3. **Multiple comparisons:** which "meaningful" verdicts survive a Holm correction within each section?

In [1]:
import os
import subprocess
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from statsmodels.stats.multitest import multipletests

RAW = Path("../data/raw")
PROC = Path("../data/processed")
pd.set_option("display.width", 200)

## 1. Attack-zone validation against Savant
Savant's swing/take leaderboard gives each pitcher's run value by attack zone (regular season, pitcher's
perspective: positive = runs saved). We assign bands ourselves (same code as notebook 03), sum
`delta_run_exp` per band, flip the sign to the pitcher's perspective, and compare.

In [2]:
rows = []
for name, pid in [("sasaki", 808963), ("yamamoto", 808967)]:
    for y in [2025, 2026]:
        d = pd.read_csv(RAW / f"{name}_statcast_mlb_{y}.csv", low_memory=False)
        d = d[d.game_type == "R"]
        half_w = (17 / 2) / 12 + 1.45 / 12
        half_h = (d.sz_top - d.sz_bot) / 2 + 1.45 / 12
        edge = np.maximum((d.plate_x / half_w).abs(), ((d.plate_z - (d.sz_top + d.sz_bot) / 2) / half_h).abs())
        d["band"] = np.select([edge < 0.67, edge < 1.33, edge < 2.0], ["heart", "shadow", "chase"], "waste")
        ours = -d.groupby("band").delta_run_exp.sum()
        sv = pd.read_csv(RAW / f"savant_swing_take_pitchers_{y}.csv")
        sv = sv[sv.player_id == pid].iloc[0]
        for band in ["heart", "shadow", "chase", "waste"]:
            rows.append({"pitcher": name, "season": y, "band": band, "ours (runs)": round(ours.get(band, 0), 2),
                         "savant (runs)": round(sv[f"runs_{band}"], 2), "diff": round(ours.get(band, 0) - sv[f"runs_{band}"], 2)})
zone_check = pd.DataFrame(rows)
zone_check.to_csv(PROC / "s6_attack_zone_validation.csv", index=False)
zone_check.pivot_table(index=["pitcher", "season"], columns="band", values="diff", sort=False)

/var/folders/sm/_ttyl33555v_1wtcfv790wcr0000gn/T/ipykernel_16023/80947452.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  d["band"] = np.select([edge < 0.67, edge < 1.33, edge < 2.0], ["heart", "shadow", "chase"], "waste")
/var/folders/sm/_ttyl33555v_1wtcfv790wcr0000gn/T/ipykernel_16023/80947452.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  d["band"] = np.select([edge < 0.67, edge < 1.33, edge < 2.0], ["heart", "shadow", "chase"], "waste")
/var/folders/sm/_ttyl33555v_1wtcfv790wcr0000gn/T/ipykernel_16023/80947452.py:9: 

/var/folders/sm/_ttyl33555v_1wtcfv790wcr0000gn/T/ipykernel_16023/80947452.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  d["band"] = np.select([edge < 0.67, edge < 1.33, edge < 2.0], ["heart", "shadow", "chase"], "waste")


band             heart  shadow  chase  waste
pitcher  season                             
sasaki   2025     0.30   -0.30  -0.01   0.00
         2026     0.22   -0.31   0.04   0.01
yamamoto 2025    -0.12    0.21  -0.17  -0.01
         2026     0.57   -0.91   0.25   0.00

In [3]:
zone_check.groupby("band").agg(mean_abs_diff=("diff", lambda v: v.abs().mean()), max_abs_diff=("diff", lambda v: v.abs().max())).round(2)

,mean_abs_diff,max_abs_diff
band,,
chase,0.12,0.25
heart,0.30,0.57
shadow,0.43,0.91
waste,0.00,0.01


## 2. Robustness
The H1 notebook (03) and the velocity notebook (05) read an `H1_VARIANT` switch. Each variant runs as a
throwaway copy and writes suffixed result tables. The committed notebooks stay on the baseline analysis.
- `k50`: hitter-xwOBA regression constant K = 50 instead of 100
- `tertiles`: three hitter tiers (league PA-weighted thirds) instead of two
- `no_tokyo`: drop Sasaki's 2025-03-19 and 2025-03-29 starts (the two high-walk "good" starts flagged in H1)

In [4]:
RUNS = [("03_command_investigation", ["k50", "tertiles", "no_tokyo"]), ("05_velocity_exploratory", ["no_tokyo"])]
with tempfile.TemporaryDirectory() as tmp:
    for nb, variants in RUNS:
        for v in variants:
            subprocess.run([sys.executable, "-m", "jupyter", "nbconvert", "--to", "notebook", "--execute", f"{nb}.ipynb",
                            "--output-dir", tmp, "--output", f"{nb}_{v}", "--ExecutePreprocessor.timeout=1800"],
                           check=True, env={**os.environ, "H1_VARIANT": v}, capture_output=True)
            print("ran", nb, v)

ran 03_command_investigation k50


ran 03_command_investigation tertiles


ran 03_command_investigation no_tokyo


ran 05_velocity_exploratory no_tokyo


In [5]:
VARIANTS = ["baseline", "k50", "tertiles", "no_tokyo"]
def sfx(v):
    return "" if v == "baseline" else f"_{v}"

rows = []
for v in VARIANTS:
    c = pd.read_csv(PROC / f"h1_contrasts{sfx(v)}.csv")
    c = c[c.contrast == "All starts − Relief"]
    for _, r in c.iterrows():
        rows.append({"variant": v, "test": r.metric, "diff": r["diff"], "ci": f"[{r.ci_lo}, {r.ci_hi}]",
                     "verdict": r.verdict if isinstance(r.verdict, str) else ""})
    e = pd.read_csv(PROC / f"h1_link_e{sfx(v)}.csv", index_col=0)
    bb, xw = e.loc["bb_pct"], e.loc["xwobacon"]
    passes = bb["diff"] >= 3 and bb["diff_lo"] > 0 and bb["std_diff"] > xw["std_diff"]
    rows.append({"variant": v, "test": "(e) BB% bad − good starts", "diff": round(bb["diff"], 2),
                 "ci": f"[{bb['diff_lo']:.2f}, {bb['diff_hi']:.2f}]",
                 "verdict": "Supported" if passes else ("Contradicted" if bb["diff_hi"] < 0 else "Not supported")})
    rows.append({"variant": v, "test": "(e) xwOBAcon bad − good starts", "diff": round(xw["diff"], 3),
                 "ci": f"[{xw['diff_lo']:.3f}, {xw['diff_hi']:.3f}]", "verdict": ""})
h1_rob = pd.DataFrame(rows)
h1_rob.to_csv(PROC / "s6_robustness_h1.csv", index=False)
wide = h1_rob.pivot_table(index="test", columns="variant", values="verdict", aggfunc="first", sort=False)[VARIANTS]
wide

variant,baseline,k50,tertiles,no_tokyo
test,,,,
(a) Putaway%,Inconclusive,Inconclusive,Inconclusive,Inconclusive
(a) 2K whiff% [stuff],Stuff not shown to decline,Stuff not shown to decline,Stuff not shown to decline,Stuff not shown to decline
(b) Expansion gap (pts),,,,
(c-i) Waste share,Inconclusive,Inconclusive,Inconclusive,Inconclusive
(c-ii) Chase rate,No stuff signal,No stuff signal,No stuff signal,No stuff signal
(d) 2K walk rate,Inconclusive,Inconclusive,Inconclusive,Inconclusive
(d) Overall BB%,,,,
(e) BB% bad − good starts,Not supported,Not supported,Not supported,Not supported
(e) xwOBAcon bad − good starts,,,,


In [6]:
h1_rob.pivot_table(index="test", columns="variant", values="diff", aggfunc="first", sort=False)[VARIANTS]

variant,baseline,k50,tertiles,no_tokyo
test,,,,
(a) Putaway%,-3.400,-3.400,-4.500,-3.400
(a) 2K whiff% [stuff],1.500,1.500,1.100,1.800
(b) Expansion gap (pts),-1.800,-1.800,-1.700,-1.300
(c-i) Waste share,-8.700,-8.700,-7.400,-9.300
(c-ii) Chase rate,-6.000,-6.000,-4.900,-5.800
(d) 2K walk rate,4.600,4.600,4.500,3.900
(d) Overall BB%,2.300,2.300,1.800,1.400
(e) BB% bad − good starts,-3.190,-3.190,-3.190,-0.090
(e) xwOBAcon bad − good starts,0.084,0.084,0.084,0.068


In [7]:
flips = wide[wide.nunique(axis=1) > 1]
print("H1 verdicts that change under any variant:", "none" if flips.empty else list(flips.index))

H1 verdicts that change under any variant: none


`k50` matches baseline exactly. That's expected, not a broken switch: K = 50 moves only a handful of low-PA batters across the median tier line, and Sasaki faced none of them. Check:

In [ ]:
for y in [2025, 2026]:
    faced = set(pd.read_csv(RAW / f"sasaki_statcast_mlb_{y}.csv", low_memory=False).batter)
    x = pd.read_csv(RAW / f"batter_xwoba_{y}.csv")
    lg = (x.est_woba * x.pa).sum() / x.pa.sum()
    tier = {}
    for K in [100, 50]:
        adj = (x.pa * x.est_woba + K * lg) / (x.pa + K)
        s = adj.sort_values()
        med = s[(x.pa[s.index].cumsum() / x.pa.sum()) >= 0.5].iloc[0]
        tier[K] = np.where(adj >= med, "High", "Low")
    moved = tier[100] != tier[50]
    print(f"{y}: batters changing tier league-wide {int(moved.sum())}, of whom Sasaki faced {int(x.player_id[moved].isin(faced).sum())}")

In [8]:
rows = []
for v in ["baseline", "no_tokyo"]:
    a = pd.read_csv(PROC / f"s5_velocity_exploratory{sfx(v)}.csv")
    a = a[(a.pitcher == "sasaki") & (a.outcome == "xwoba")]
    for _, r in a.iterrows():
        rows.append({"variant": v, "sample": "all starts", "predictor": r.predictor, "starts": r.starts,
                     "slope per mph": r["slope per mph"], "slope CI": r["slope CI"], "r": r.r, "r CI": r["r CI"]})
velo_rob = pd.DataFrame(rows)
velo_rob.to_csv(PROC / "s6_robustness_velocity.csv", index=False)
velo_rob

,variant,sample,predictor,starts,slope per mph,slope CI,r,r CI
0,baseline,all starts,raw FF velo,31,-0.031,"[-0.048, -0.017]",-0.54,"[-0.72, -0.31]"
1,baseline,all starts,season-centered velo,31,-0.038,"[-0.065, -0.017]",-0.49,"[-0.70, -0.24]"
2,no_tokyo,all starts,raw FF velo,29,-0.032,"[-0.050, -0.017]",-0.54,"[-0.72, -0.31]"
3,no_tokyo,all starts,season-centered velo,29,-0.043,"[-0.072, -0.020]",-0.52,"[-0.73, -0.25]"


## 3. Multiple comparisons (Holm, α = 0.05, within each section)
Two-sided bootstrap p-values come from the same draws as the CIs. A "claimed" result is one the pre-registered
rule called Supported / Meaningful. Holm controls the chance of any false positive within a family.

In [9]:
fam = []
h1 = pd.read_csv(PROC / "h1_contrasts.csv")
h1 = h1[(h1.contrast == "All starts − Relief") & h1.verdict.notna()]
for _, r in h1.iterrows():
    fam.append({"family": "S2 / H1", "test": r.metric, "p_boot": r.p_boot, "claimed": r.verdict == "Supported"})
x = pd.read_csv(PROC / "h1_expansion.csv").iloc[0]
fam.append({"family": "S2 / H1", "test": x.test, "p_boot": x.p_boot, "claimed": x.ci_lo > 0})
e = pd.read_csv(PROC / "h1_link_e.csv", index_col=0)
fam.append({"family": "S2 / H1", "test": "(e) BB% bad − good starts", "p_boot": e.loc["bb_pct", "p_boot"], "claimed": False})

s1 = pd.read_csv(PROC / "s1_contrasts.csv")
s1 = s1[s1.verdict.notna() & s1.p_boot.notna()]
for _, r in s1.iterrows():
    fam.append({"family": "S1 / H3-H4", "test": f"{r.contrast} — {r.metric}", "p_boot": r.p_boot, "claimed": r.verdict == "Meaningful"})

s3 = pd.read_csv(PROC / "s3_pvalues.csv")
for _, r in s3.iterrows():
    fam.append({"family": "S3 / H2-H5", "test": r.test, "p_boot": r.p_boot, "claimed": bool(r.meaningful)})
fam = pd.DataFrame(fam)

out = []
for f, g in fam.groupby("family", sort=False):
    reject = multipletests(g.p_boot.values, alpha=0.05, method="holm")[0]
    out.append(g.assign(holm_reject=reject, tests_in_family=len(g)))
mc = pd.concat(out)
mc.to_csv(PROC / "s6_multiple_comparisons.csv", index=False)
summary = mc.groupby("family", sort=False).agg(tests=("test", "size"), claimed=("claimed", "sum"),
                                                 claimed_surviving_holm=("holm_reject", lambda r: int((r & mc.loc[r.index, "claimed"]).sum())),
                                                 expected_false_positives_at_05=("test", lambda t: round(0.05 * len(t), 1)))
summary

,tests,claimed,claimed_surviving_holm,expected_false_positives_at_05
family,,,,
S2 / H1,7,1,1,0.4
S1 / H3-H4,31,16,12,1.6
S3 / H2-H5,12,3,1,0.6


In [10]:
mc[mc.claimed].sort_values(["family", "p_boot"])[["family", "test", "p_boot", "holm_reject"]]

,family,test,p_boot,holm_reject
7,S1 / H3-H4,FF: 2026 start − 2025 start — velo,0.000100,True
12,S1 / H3-H4,FF: 2025 relief − 2025 start — velo,0.000100,True
17,S1 / H3-H4,FF: 2026 relief − 2026 start — velo,0.000100,True
24,S1 / H3-H4,FO: 2025 relief − 2025 start — HB,0.000100,True
28,S1 / H3-H4,FS: 2026 relief − 2026 start — IVB,0.000100,True
31,S1 / H3-H4,Splitter change: 2026 FS − 2025 FO (starts) — ...,0.000100,True
32,S1 / H3-H4,Splitter change: 2026 FS − 2025 FO (starts) — IVB,0.000100,True
33,S1 / H3-H4,Splitter change: 2026 FS − 2025 FO (starts) — HB,0.000100,True
37,S1 / H3-H4,Gap: 2026 start FS − 2025 start FO — FF-splitt...,0.000100,True
30,S1 / H3-H4,FS: 2026 relief − 2026 start — chase,0.000142,True
